# pyMAISE Setup Guide

This notebook will help you install pyMAISE locally. **Read everything before you start — don't just run cells blindly.**

---

## Before You Open This Notebook

You need to create a dedicated Python environment **in your terminal** first. Do NOT skip this step — installing pyMAISE into your default Python will break other packages.

### If You Have Anaconda or Miniconda

Open a terminal (Mac/Linux) or Anaconda Prompt (Windows) and run:

```bash
conda create -n pymaise python=3.12 jupyter -y
conda activate pymaise
python -m ipykernel install --user --name pymaise --display-name "Python 3.12 (pyMAISE)"
```

### If You Don't Have Conda

Install Python 3.12 first:
- **Windows:** Download from https://www.python.org/downloads/ — check "Add Python to PATH"
- **Mac:** `brew install python@3.12`
- **Linux:** `sudo apt update && sudo apt install python3.12 python3.12-venv`

Then:

```bash
# Windows
py -3.12 -m venv pymaise-env
pymaise-env\Scripts\activate
pip install jupyter
python -m ipykernel install --user --name pymaise --display-name "Python 3.12 (pyMAISE)"

# Mac/Linux
python3.12 -m venv pymaise-env
source pymaise-env/bin/activate
pip install jupyter
python -m ipykernel install --user --name pymaise --display-name "Python 3.12 (pyMAISE)"
```

### Then:

1. Launch Jupyter from within the activated environment: `jupyter notebook`
2. Open this notebook
3. Go to **Kernel → Change kernel → Python 3.12 (pyMAISE)**
4. Now run Step 1 below

---

### Not sure if you have Conda?

In [ ]:
import shutil
conda_path = shutil.which("conda")
if conda_path:
    print(f"Conda found at: {conda_path}")
    print("Use the Conda instructions above.")
else:
    print("Conda not found.")
    print("Use the non-Conda instructions above, or install Miniconda from:")
    print("https://docs.anaconda.com/miniconda/install/")

---

## Step 1: Verify Your Environment

This checks that you're running the right Python in the right environment.

In [ ]:
import sys

major, minor, micro = sys.version_info[:3]
print(f"Python version: {major}.{minor}.{micro}")
print(f"Executable:     {sys.executable}")
print()

ok = True

if major == 3 and minor == 12:
    print("✅ Python 3.12 — correct version.")
else:
    print(f"❌ You need Python 3.12, but you're running {major}.{minor}.")
    print("   Did you switch to the 'Python 3.12 (pyMAISE)' kernel?")
    print("   Go to Kernel → Change kernel and select it.")
    ok = False

if 'pymaise' in sys.executable.lower():
    print("✅ Running inside the pymaise environment.")
else:
    print(f"⚠️  Your Python path doesn't contain 'pymaise': {sys.executable}")
    print("   This might mean you're in the wrong environment.")
    print("   Make sure you selected the 'Python 3.12 (pyMAISE)' kernel.")
    ok = False

if ok:
    print()
    print("Good to go — proceed to Step 2.")

---

## Step 2: Install pyMAISE

Only run this after Step 1 shows ✅ for both checks.

**What this does:** pyMAISE's PyPI package has a dependency (`tensorflow[and-cuda]`) that requires NVIDIA GPU libraries. These don't exist on Mac, and many Windows/Linux machines don't have them either. The install below works around this by installing TensorFlow and all other dependencies first (without the CUDA extras), then installing pyMAISE without its automatic dependency resolution.

This will take a few minutes — TensorFlow is a large download.

In [ ]:
import sys
import platform

# Safety checks
major, minor = sys.version_info[:2]
if not (major == 3 and minor == 12):
    print(f"⚠️  Wrong Python version ({major}.{minor}). Go back to Step 1.")
elif 'pymaise' not in sys.executable.lower():
    print(f"⚠️  Doesn't look like the pymaise environment. Go back to Step 1.")
else:
    print("[1/4] Upgrading pip and build tools...")
    !{sys.executable} -m pip install --upgrade pip setuptools wheel --quiet

    print("[2/4] Installing TensorFlow and core ML dependencies...")
    !{sys.executable} -m pip install tensorflow "tensorflow-probability[tf]>=0.25.0" --quiet

    print("[3/4] Installing remaining dependencies...")
    !{sys.executable} -m pip install keras-tuner scikeras scikit-optimize matplotlib graphviz pydot tqdm xarray pandas scikit-learn --quiet

    print("[4/4] Installing pymaise-dev...")
    !{sys.executable} -m pip install pymaise-dev --no-deps --quiet

    print()
    print("✅ Installation complete! Run Step 3 to verify.")
    
    if platform.system() == 'Darwin':
        print()
        print("Note for Mac users: Apple GPU acceleration is not available with")
        print("TensorFlow >=2.18. Everything runs on CPU, which is fine for the benchmarks.")

### Optional: Graphviz System Binary

pyMAISE uses Graphviz to draw neural network architecture diagrams. The Python package is installed above, but it also needs the actual Graphviz program on your system. **This is optional** — everything works without it except the network plots.

Install in your terminal (not here):
- **Mac:** `brew install graphviz`
- **Windows:** Download from https://graphviz.org/download/ and add to PATH
- **Linux:** `sudo apt install graphviz`

---

## Step 3: Verify Installation

In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"Path:   {sys.executable}")
print()

checks = []

packages = [
    ("pyMAISE",        "import pyMAISE as mai"),
    ("TensorFlow",     "import tensorflow as tf; v = tf.__version__"),
    ("scikit-learn",   "import sklearn; v = sklearn.__version__"),
    ("xarray",         "import xarray; v = xarray.__version__"),
    ("keras-tuner",    "import keras_tuner; v = keras_tuner.__version__"),
    ("scikit-optimize", "import skopt; v = skopt.__version__"),
    ("scikeras",       "import scikeras; v = scikeras.__version__"),
    ("matplotlib",     "import matplotlib; v = matplotlib.__version__"),
    ("pandas",         "import pandas; v = pandas.__version__"),
]

for name, code in packages:
    try:
        v = None
        exec(code)
        version_str = f" {v}" if v else ""
        print(f"  ✅ {name}{version_str}")
        checks.append(True)
    except Exception as e:
        print(f"  ❌ {name}: {e}")
        checks.append(False)

# Graphviz binary (optional)
import shutil
if shutil.which('dot'):
    print(f"  ✅ Graphviz system binary")
else:
    print(f"  ⚠️  Graphviz system binary not found (optional — see note above)")

print()
if all(checks):
    print("🎉 All checks passed! Proceed to Step 4.")
else:
    print("❌ Some packages failed. Re-run Step 2 or check the errors above.")

---

## Step 4: Smoke Test — Load the MIT Reactor Dataset

If this runs, pyMAISE is fully functional and you're ready for the assignment.

In [ ]:
import pyMAISE as mai
from pyMAISE.datasets import load_MITR

settings = mai.init(
    problem_type=mai.ProblemType.REGRESSION,
    cuda_visible_devices="-1"
)

data, inputs, outputs = load_MITR()

print(f"Dataset loaded successfully!")
print(f"  Inputs shape:  {inputs.shape}")
print(f"  Outputs shape: {outputs.shape}")
print()
print("✅ pyMAISE is working! You're ready for the MIT reactor notebook.")

---

## Troubleshooting

**"I see the wrong Python version in Step 1"**

Go to Kernel → Change kernel → Python 3.12 (pyMAISE). If you don't see that option, go back to your terminal and make sure you ran the `ipykernel install` command from the instructions at the top.

**Step 2 shows warnings about dependency conflicts**

Lines starting with `ERROR: pip's dependency resolver...` after a `Successfully installed` message are warnings, not actual failures. As long as Step 3 shows all ✅, you're fine.

**TensorFlow shows warnings about GPU/CUDA/Metal**

Normal and harmless. TensorFlow falls back to CPU, which is fine for these benchmarks.

**"ModuleNotFoundError: No module named 'pyMAISE'"**

This usually means the install went to the wrong environment. Check that Step 1 shows your executable path containing `pymaise`. If it doesn't, your kernel is pointing to the wrong Python. Switch kernels and try again.

**Everything installed but Step 4 crashes**

Try Kernel → Restart and run Step 4 again. Some packages need a fresh kernel after installation.

**"I want to start over"**

That's the beauty of isolated environments — you can delete and recreate them cleanly:
```bash
conda deactivate
conda remove -n pymaise --all -y
```
Then follow the instructions at the top of this notebook again.